In [2]:
# ------------------------------------------------------------------------------
# Setup and Path Validation
# ------------------------------------------------------------------------------
from app.core.tools.catalog_tool import catalog_lookup, CATALOG_PATH
import os
import json
from pathlib import Path
import sys

# Ensure we can import catalog_tool
HERE = Path.cwd().resolve()
print("Running from:", HERE)


def is_repo_root(p: Path) -> bool:
    # Prefer strong layout markers to avoid accidentally selecting /scripts as root
    return (
        (p / "pyproject.toml").exists()
        or (p / ".git").exists()
        or ((p / "app").exists() and (p / "scripts").exists() and (p / "data").exists())
        or ((p / "knowledge").exists() and (p / "data").exists())
        or (p / "main.py").exists()
    )

# Adjust this to point to your repo root so 'data/catalog.json' is reachable
PROJECT_ROOT = next(p for p in [HERE, *HERE.parents] if is_repo_root(p))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Catalog Path: {CATALOG_PATH}")
print(f"Catalog exists? {CATALOG_PATH.exists()}")

Running from: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/scripts
Project Root: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot
Catalog Path: /Users/damiangarayalde/Work - Github Repos/AI/sneil_chatbot/data/catalog.json
Catalog exists? True


In [3]:
# ------------------------------------------------------------------------------
# Inspect Catalog Data
# ------------------------------------------------------------------------------
# Let's see what we are working with
if CATALOG_PATH.exists():
    data = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
    # Note: your ingest script uses "items", tool uses "products"
    items = data.get("items", [])
    print(f"Total items in catalog: {len(items)}")

    # Check first 2 items to see structure
    for item in items[:2]:
        print(
            f"- SKU: {item['sku']} | Model: {item['model']} | Family: {item['family']}")
else:
    print("❌ ERROR: catalog.json not found. Run ingest_raw_catalog.ipynb first.")


Total items in catalog: 71
- SKU: C019 | Model: C300 | Family: TPMS
- SKU: C022 | Model: C260 Externo | Family: TPMS


In [4]:
# ------------------------------------------------------------------------------
#  Helper Function for Pretty Printing Results
# ------------------------------------------------------------------------------

def test_query(query, family=None, k=3):
    print(f"\nSearching for: '{query}' | Family Filter: {family}")
    try:
        results = catalog_lookup(query, product_family=family, k=k)
        print(f"Found {results['count']} matches (showing top {k}):")
        for i, match in enumerate(results['matches'], 1):
            price = match.get('price', {}).get('ars', 'N/A')
            print(
                f"  {i}. [{match.get('sku')}] {match.get('model')} - ${price} {results['currency']}")
    except Exception as e:
        print(f"❌ Search failed: {e}")

In [5]:
# ------------------------------------------------------------------------------
# 5) Functional Smoke Tests
# ------------------------------------------------------------------------------


# Test 1: Exact SKU lookup
# Replace 'SKU_HERE' with a real SKU from your catalog.json
test_query("SKU_HERE")

# Test 2: Model name lookup (Fuzzy/Token match)
test_query("HDK 2200")

# Test 3: Family filtering (Should only return TPMS)
test_query("sensor", family="TPMS")

# Test 4: Broad search (Should return multiple items up to k=5)
test_query("aire", k=5)


Searching for: 'SKU_HERE' | Family Filter: None
Found 0 matches (showing top 3):

Searching for: 'HDK 2200' | Family Filter: None
Found 3 matches (showing top 3):
  1. [C04312] HDK2200 12v (para motorhomes) - $1822012.62 ARS
  2. [C04312_E] HDK2200 12v (para camiones) - $1822012.62 ARS
  3. [C04324] HDK2200 24v - $1822012.62 ARS

Searching for: 'sensor' | Family Filter: TPMS
❌ Search failed: 'product_family'

Searching for: 'aire' | Family Filter: None
Found 2 matches (showing top 5):
  1. [S00004] Instalacion Camion -Aire Acondicionado/Caldera- - $None ARS
  2. [S001] Instalacion Utilitario/Motorhome -Aire Acondicionado/Caldera- - $None ARS


In [6]:
# ------------------------------------------------------------------------------
# 6) Edge Case Testing
# ------------------------------------------------------------------------------

print("\n--- Edge Case Testing ---")

# Test: Case Insensitivity
test_query("tpms")
test_query("TPMS")

# Test: Non-existent product
test_query("Flux Capacitor 9000")

# Test: Empty Query (behavior check)
test_query("")


--- Edge Case Testing ---

Searching for: 'tpms' | Family Filter: None
Found 3 matches (showing top 3):
  1. [TP009] Repuesto de sensor externo TPMS - $18335.86 ARS
  2. [TP010] Repuesto de sensor interno TPMS - $13580.37 ARS
  3. [TP012] Valvula para sensor interno TPMS - $6008.19 ARS

Searching for: 'TPMS' | Family Filter: None
Found 3 matches (showing top 3):
  1. [TP009] Repuesto de sensor externo TPMS - $18335.86 ARS
  2. [TP010] Repuesto de sensor interno TPMS - $13580.37 ARS
  3. [TP012] Valvula para sensor interno TPMS - $6008.19 ARS

Searching for: 'Flux Capacitor 9000' | Family Filter: None
Found 0 matches (showing top 3):

Searching for: '' | Family Filter: None
Found 3 matches (showing top 3):
  1. [C019] C300 - $188300.64 ARS
  2. [C022] C260 Externo - $77616.67 ARS
  3. [C022+1] C260 Externo + auxilio - $89258.6 ARS
